# Faruq-v3 IGEM breadth screening

Seed-42 validation-only screen for **IGEM1**. The transfer keeps the native YOLO26 box branch and adds a class-aware reference/mask-guided enhancement path for classification. Test is never extracted or opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/igem-classification-guidance-screening'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan GPU'
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt','experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json'))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
CONTROL=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as a: a.extractall('/content',filter='data')
GROUPED=DATA_ROOT/'faruq_grouped_summary.json'; assert GROUPED.is_file(); assert not (DATA_ROOT/'test').exists()
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-igem-screening-v1'
print(torch.cuda.get_device_name(0),OUTPUT)


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_igem.py'],cwd=REPO,check=True)


In [ ]:
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_igem_screening','--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),'--control-summary',str(CONTROL),'--d0-checkpoint',str(D0),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('RUN:',' '.join(cmd),flush=True); subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
import json,pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/igem_seed42_screening.json'; r=json.loads(SUMMARY.read_text())
assert r['test_opened'] is False and r['test_images_accessed'] is False
rows=[{'model':k,**v} for k,v in r['controls'].items()]+[{'model':'IGEM1',**r['candidate']['IGEM1']}]
display(pd.DataFrame(rows))
print('Decision:',r['decision']); print('Summary:',SUMMARY)
